# Generation checks

Full pipeline test from a raw guest question through to a generated answer, with an LLM doing
the query understanding as well as the final reply -- two calls per question, both fully
printed:

1. **Understand** -- one structured-output LLM call reads the question and returns intent
   (`menu` vs `faq`) plus retrieval filters (dietary, price ceiling, allergens to exclude).
   This replaces `01_retrieval_checks.ipynb`'s regex-based `parse_constraints()` and this
   notebook's earlier top-1-`item_type` `classify_intent()` -- both were explicitly called out
   as heuristics to swap for an LLM call once one was available.
2. **Retrieve** -- the rest of `01`'s pipeline, unchanged: hybrid `alpha=0.75` -> allergen
   exclude (union of `allergens_contains`/`allergens_may_contain`) -> rerank `rerank-v3.5` ->
   gate `0.15`.
3. **Generate** -- a grounding-only system prompt, with a tone instruction picked from the
   understanding step's intent: **precise and literal for menu facts**
   (price/allergens/nutrition must stay exact), **warm and conversational for FAQ answers**
   (house-policy copy can be phrased more naturally). This used to be a `temperature` value
   per intent; `gemini-3.8-flash` (Gemini's 3.x Flash line) silently ignores `temperature` /
   `top_p` / `top_k` now -- Google's own guidance is to steer behavior through system
   instructions instead, which is what section 5 does.

Every test cell prints both full prompts (understanding and generation) and both LLM outputs,
so the whole path from question to answer is inspectable, not just the final text.

Runs against the live Weaviate collection and the live LLM API -- nothing is mocked.

Prerequisites: the collection is loaded (`scripts/load_knowledge_base.py`), and `.env` holds
`WEAVIATE_URL` / `WEAVIATE_API_KEY`, `EMBEDDING_API_KEY` (Cohere, for reranking), and
`LLM_API_KEY` (`LLM_PROVIDER=google`, `LLM_MODEL=gemini-3.8-flash` by default) --
`00_environment_check.ipynb` verifies all three. Reranking needs the Cohere key; a trial key
is rate-limited, so some cells may fall back to hybrid order (see `01`). Without `LLM_API_KEY`
the understanding step falls back to `{intent: "menu", no filters}` and generation is skipped,
so retrieval-only checks still run.

## Pipeline overview

What happens between a guest's question and the answer -- every external API call, in order.
Two Gemini (LLM) calls and up to two Cohere calls per question (the embedding call is made by
Weaviate itself, mid-retrieval, and isn't shown as a separate box the way the direct rerank
call is -- see section 4's note on why it can't be measured from here).

Node names match the actual function each step calls -- see the matching section below for
what each one does.

```mermaid
flowchart LR
    Q[Guest question] --> U["understand_query()<br/>LLM 1: Gemini"]
    U --> F[build_filter]
    F --> R["retrieve()<br/>Weaviate hybrid"]
    R --> EX{allergens_exclude?}
    EX -- yes --> DROP[drop excluded rows]
    EX -- no --> KEEP[keep all rows]
    DROP --> RR
    KEEP --> RR["rerank()<br/>Cohere"]
    RR --> GATE{"top score >= 0.15?"}
    GATE -- no --> NC["CONTEXT = no match"]
    GATE -- yes --> CTX[build_context]
    NC --> PROMPT
    CTX --> PROMPT[build_user_prompt]
    PROMPT --> TEMP[tone_for intent]
    TEMP --> GEN["call_llm()<br/>LLM 2: Gemini"]
    GEN --> ANS[Answer to guest]
```

## 1. Connect

Loads `.env`, connects to Weaviate Cloud (same as `01`), and reads the LLM credentials
(`00_environment_check.ipynb` verifies these are valid). The client stays open for the whole
notebook, including the demo cell at the end -- re-run this cell to reconnect if the kernel
session drops. Closes any `client` left over from a previous run of this same cell first --
otherwise reconnecting in an already-running kernel leaks the old connection's sockets until
Python's garbage collector eventually finalizes them, which is what an "unclosed
`<ssl.SSLSocket ...>`" `ResourceWarning` after a reconnect actually is.

In [ ]:
import contextlib
import json
import math
import os
import random
import time
import urllib.error
import urllib.request

import weaviate
from dotenv import find_dotenv, load_dotenv
from weaviate.classes.init import Auth
from weaviate.classes.query import Filter, FilterReturn, MetadataQuery

load_dotenv(find_dotenv(usecwd=True))

PROVIDER = os.environ.get("EMBEDDING_PROVIDER", "cohere").lower()
COHERE_KEY = os.environ.get("EMBEDDING_API_KEY", "")
_hdr = "X-OpenAI-Api-Key" if PROVIDER == "openai" else "X-Cohere-Api-Key"

# Re-running this cell in an already-running kernel would otherwise leave the previous
# client's sockets open until Python's garbage collector gets around to them -- that's what
# an "unclosed <ssl.SSLSocket ...>" ResourceWarning after a reconnect is. Close it first.
# globals().get(...) (rather than referencing `client` directly) means this doesn't depend on
# `client` already existing in this kernel session.
_previous_client = globals().get("client")
if _previous_client is not None:
    with contextlib.suppress(Exception):
        _previous_client.close()

client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=Auth.api_key(os.environ["WEAVIATE_API_KEY"]),
    headers={_hdr: COHERE_KEY},
)
kb = client.collections.get("KnowledgeBase")
print("connected -", kb.aggregate.over_all(total_count=True).total_count, "objects")

LLM_PROVIDER = os.environ.get("LLM_PROVIDER", "google").lower()
LLM_MODEL = os.environ.get("LLM_MODEL", "gemini-3.8-flash")
LLM_API_KEY = os.environ.get("LLM_API_KEY", "")

print(f"LLM_PROVIDER = {LLM_PROVIDER}")
print(f"LLM_MODEL    = {LLM_MODEL}")
print(f"LLM_API_KEY  = {'set (' + str(len(LLM_API_KEY)) + ' chars)' if LLM_API_KEY else 'NOT SET'}")
if LLM_PROVIDER != "google":
    print(
        f"warning: only the google/Gemini REST call is wired into this notebook, "
        f"LLM_PROVIDER={LLM_PROVIDER!r} will not be called"
    )
if not LLM_API_KEY:
    print("warning: LLM_API_KEY not set -- understanding/generation cells will use fallbacks")

## 2. LLM call helper

One thin wrapper around the Gemini `generateContent` REST endpoint (same style
`00_environment_check.ipynb` uses for its credential check), shared by both LLM calls this
notebook makes: query understanding (structured JSON, via `response_schema`) and answer
generation (free text). Keeping one call site means both share the same error handling,
timeout, retry behavior, and usage accounting instead of drifting apart.

`call_llm()` returns `{"text": ..., "usage": {...}}`, not just the text -- Gemini's response
carries its own token counts (`usageMetadata`), and reading them straight from the API is the
only way to know what a call actually cost, rather than estimating from string length. This
is what lets `show_answer()` (section 8) report exactly how many tokens each question
consumed, split between the understanding call and the generation call, plus a running total
for the whole notebook session.

`call_llm()` takes no `temperature` parameter -- an earlier version of this notebook did, but
Google confirmed `temperature`/`top_p`/`top_k` are silently ignored on the entire Gemini 3.x
Flash line (`gemini-3.8-flash` included): the API accepts them, returns `200`, and does
nothing with them, and a future model generation will return an HTTP 400 for passing them at
all. Section 5 replaces the per-intent temperature with a per-intent tone instruction folded
into the system prompt instead, which is Google's own migration guidance for this deprecation.

Two layers of rate-limit handling, not just one:

- **Proactive pacing** -- `_pace_llm_call()` blocks just long enough before every request to
  keep the call rate under `LLM_MAX_RPM` -- a plain constant, not an env var: it's a
  notebook-local tuning knob (how fast *this run* hits the API), not shared config like
  `LLM_MODEL`/`LLM_API_KEY` that a future `app/` or a teammate's key would also need. Edit the
  constant directly to match your key's actual quota (check Google AI Studio / Cloud
  Console). Each question costs two calls (understand + generate), and the test cells fire
  several questions back-to-back, so without pacing a "Run All" can burst 25-30 calls in a
  few seconds -- comfortably over a free-tier per-minute cap. Spacing calls out to stay under
  the limit is strictly better than hitting 429 and backing off, since a 429 still costs a
  round trip for nothing.
- **Reactive retry**, for whatever pacing doesn't prevent (a burst from another process on the
  same key, a transient 5xx, a dropped connection): 429/5xx/connection errors retry with
  **full-jitter exponential backoff** -- each attempt waits a random amount up to a cap that
  doubles every attempt, rather than a fixed or purely exponential delay, so that several
  calls failing at once don't all retry in lockstep and re-collide. A `Retry-After` header,
  when the API sends one, overrides the computed delay -- the server's own estimate beats a
  guess. A non-retryable error (400 bad request, 401/403 auth) raises immediately, since
  retrying won't fix a malformed request or a bad key; a retryable error raises its original
  exception once retries are exhausted, so the final traceback still shows the real HTTP
  status instead of a wrapper.

In [ ]:
RETRYABLE_HTTP_CODES = {408, 429, 500, 502, 503, 504}
MAX_LLM_RETRIES = 5
BASE_DELAY_S = 1.0
MAX_DELAY_S = 20.0
LLM_MAX_RPM = 15.0  # client-side pacing cap -- edit to match your key's actual quota

_last_llm_call_at = 0.0


def _zero_usage() -> dict:
    """A fresh {prompt,completion,total: 0} usage dict, for calls that never hit the API."""
    return {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}


def _http_error_detail(e: urllib.error.HTTPError) -> str:
    """Read and return a truncated response body from an HTTPError, then close it.

    A 429's body is where the actual reason lives -- Gemini's error responses include a
    QuotaFailure naming the specific quota exceeded (e.g. a per-minute vs a per-day quota ID),
    which the bare status code and reason phrase never show. Reading it here means every
    retry/failure message can show *which* limit was hit instead of just that one was.
    """
    try:
        raw = e.read().decode("utf-8", errors="replace")
    except OSError:
        raw = ""
    e.close()
    return raw[:400].replace("\n", " ")


def _pace_llm_call() -> None:
    """Block just long enough to keep call_llm() under LLM_MAX_RPM requests/minute."""
    global _last_llm_call_at
    min_interval = 60.0 / LLM_MAX_RPM
    wait = min_interval - (time.monotonic() - _last_llm_call_at)
    if wait > 0:
        time.sleep(wait)
    _last_llm_call_at = time.monotonic()


def call_llm(
    system_prompt: str,
    user_prompt: str,
    response_schema: dict | None = None,
) -> dict:
    """POST to the Gemini generateContent endpoint.

    Returns {"text": str, "usage": {"prompt_tokens", "completion_tokens", "total_tokens"}} --
    the usage figures come straight from the API's own usageMetadata, not an estimate. Pass
    response_schema to force structured JSON output (used by understand_query()); omit it for
    a free-text reply (used by the generation step). No temperature/top_p/top_k -- Gemini's
    3.x Flash line silently ignores them; steer output via system_prompt wording instead
    (section 5's tone_for()). Paces itself under LLM_MAX_RPM before every attempt, and retries
    transient failures (429, 5xx, dropped connections) with full-jitter exponential backoff,
    honoring a Retry-After header when present. Every failed attempt prints the response
    body's detail (see _http_error_detail()) -- a 429's body names the specific quota that was
    exceeded, which the bare status code never does, and that's the only way to tell a
    per-minute throttle (clears in seconds) from a daily/project quota (clears on a 24h
    boundary, not by retrying). Raises immediately on a non-retryable error, or once retries
    are exhausted.
    """
    url = (
        f"https://generativelanguage.googleapis.com/v1beta/models/{LLM_MODEL}:generateContent"
        f"?key={LLM_API_KEY.strip()}"
    )
    generation_config: dict = {}
    if response_schema is not None:
        generation_config["responseMimeType"] = "application/json"
        generation_config["responseSchema"] = response_schema
    body = json.dumps(
        {
            "systemInstruction": {"parts": [{"text": system_prompt}]},
            "contents": [{"role": "user", "parts": [{"text": user_prompt}]}],
            "generationConfig": generation_config,
        }
    ).encode()

    delay_cap = BASE_DELAY_S
    for attempt in range(1, MAX_LLM_RETRIES + 1):
        _pace_llm_call()
        req = urllib.request.Request(
            url, data=body, headers={"Content-Type": "application/json"}, method="POST"
        )
        retry_after: str | None = None
        detail = ""
        try:
            with urllib.request.urlopen(req, timeout=60) as resp:
                data = json.load(resp)
            parts = data["candidates"][0]["content"]["parts"]
            text = " ".join(p.get("text", "") for p in parts).strip()
            meta = data.get("usageMetadata", {})
            usage = {
                "prompt_tokens": meta.get("promptTokenCount", 0),
                "completion_tokens": meta.get("candidatesTokenCount", 0),
                "total_tokens": meta.get("totalTokenCount", 0),
            }
            return {"text": text, "usage": usage}
        except urllib.error.HTTPError as e:
            retryable = e.code in RETRYABLE_HTTP_CODES
            retry_after = e.headers.get("Retry-After") if retryable else None
            label = f"HTTP {e.code} {e.reason}"
            detail = _http_error_detail(e)
            if not retryable or attempt == MAX_LLM_RETRIES:
                print(f"   [LLM {label}, giving up -- {detail}]")
                raise
        except urllib.error.URLError as e:
            label = f"connection error ({e.reason})"
            if attempt == MAX_LLM_RETRIES:
                raise

        if retry_after:
            try:
                wait = float(retry_after)
            except ValueError:
                wait = random.uniform(0, delay_cap)
        else:
            wait = random.uniform(0, delay_cap)
        suffix = f"  {detail}" if detail else ""
        print(f"   [LLM {label}; retrying in {wait:.1f}s ({attempt}/{MAX_LLM_RETRIES})]{suffix}")
        time.sleep(wait)
        delay_cap = min(delay_cap * 2, MAX_DELAY_S)

    raise RuntimeError("unreachable")  # the loop above always returns or raises

## 3. Query understanding (LLM)

Replaces two heuristics with one LLM call: `01`'s regex `parse_constraints()` (dietary/price
wording, allergy wording) and this notebook's earlier `classify_intent()` (guessing intent
from whichever row retrieval happened to rank first). `understand_query()` asks the model
directly, before any retrieval happens, for:

- `intent` -- `menu` or `faq`.
- `dietary` -- `vegan`/`vegetarian`/`none`. `none` on purpose for a general availability
  question ("do you have vegan options") -- `01` learned that hard-filtering those loses FAQ
  rows, since FAQ rows carry no `dietary_tags` of their own.
- `price_max_gbp` -- only for a firm ceiling ("under £8"), not vague wording ("affordable").
- `allergens_exclude` -- canonical allergen names, constrained by `response_schema`'s `enum`
  to the same vocabulary `01` used for its regex synonym map, so the model can't return a
  value the corpus wouldn't recognise.
- `search_query` -- the question with dietary/price/allergy wording stripped out, since that's
  already handled by `dietary`/`price_max_gbp`/`allergens_exclude` above. Retrieval and rerank
  use this instead of the raw question (section 4's `search()`), so "vegan" and "under £6"
  stop diluting the vector match for the word that actually identifies the dish.
- `category_hint` -- zero or more of the menu's own category names, guessed from course-type
  language ("starter", "main", "dessert", "drink"). Constrained by `response_schema`'s `enum`
  to `MENU_CATEGORIES`, fetched live from the collection below rather than a hardcoded
  synonym list, so it can't drift from the real menu. `expand_category_hint()` then adds
  sibling categories -- ones sharing an immediate parent in `category_path` -- so getting one
  sibling right pulls in the rest without expecting the model to enumerate every one from
  memory. Folded into the search text as a soft signal (section 4), not a hard filter -- a
  wrong guess should never be able to hide the right dish, only fail to help find it.
- `gluten_free_only` -- true ONLY when the guest asks for the restaurant's own curated
  gluten-free menu/section by name ("what's on your gluten-free menu", "gluten-free options").
  This is a positive filter on `is_gluten_free_listed`, separate from `allergens_exclude`: a
  guest describing an actual allergy ("I'm coeliac", "no gluten please") should still get
  `allergens_exclude` populated too (the safety exclusion), but a guest just browsing the
  named section doesn't need every gluten-containing dish removed from consideration first.
- `kcal_max` -- a number for a calorie ceiling, whether firm ("under 500 calories") or
  qualitative ("a low-calorie main" -> use a sensible reference like 500). null otherwise.
- `protein_min_g` -- a number for a protein floor, whether firm ("at least 20g protein") or
  qualitative ("a high-protein dish" -> use a sensible reference like 20). null otherwise.
  Unlike `price_max_gbp`, qualitative wording is honoured here with a reasonable default
  rather than left null, since "high protein" carries real meaning a guest expects acted on,
  the way "affordable" doesn't for price.
- `alcohol_free` -- true ONLY when the guest explicitly wants a non-alcoholic / alcohol-free
  drink. A positive filter on `abv_percent` being unset.

A live worked example: **"a vegan starter under £6" returned NO CONFIDENT MATCH** the first
time this notebook ran that question, even though the corpus does have exactly one vegan
starter under £6 (an oyster + shiitake mushroom bao bun) -- `data/knowledge_base.json` has no
category literally called "starters", so the raw question's vector match against 39 other
vegan-and-cheap-but-irrelevant rows (mostly drinks and desserts) drowned out the one real
match. Adding `search_query` + `category_hint` alone wasn't enough on the next run either:
the model guessed `lighter bites` and `big flavour bites` but not `bao buns` -- the category
the real match is actually in. All four (`bao buns`, `big flavour bites`, `gyoza`,
`lighter bites`) turn out to share the same `category_path` parent, `sides` -- this menu's
own closest thing to a starters section, just never labelled that anywhere. (Confusingly,
`sides` is *also* used as its own unrelated leaf category, but only under `gluten free`'s
menu section -- nothing to do with this group; a corpus vocabulary quirk in its own right.)
`expand_category_hint()` reads that sibling relationship straight from `category_path`, so
the two correct guesses now pull `bao buns` in too.

This call has no temperature setting -- `gemini-3.8-flash` ignores it either way (section 2).
What actually keeps this extraction reliable is `response_schema` itself: it constrains the
output's structure and every enum-typed field regardless of sampling settings, which is a
stronger guarantee than `temperature=0` ever was.

In [ ]:
NUT_ALLERGENS = {
    "peanuts",
    "tree nuts",
    "almond nuts",
    "walnuts",
    "hazelnuts",
    "pecan nuts",
    "pistachios",
    "brazil nuts",
    "cashew nuts",
    "macadamia nuts",
}
ALLERGEN_VOCAB: dict[str, set[str]] = {
    "peanut": {"peanuts"},
    "nut": NUT_ALLERGENS,
    "gluten": {"cereals containing gluten", "wheat", "barley", "oats", "rye"},
    "wheat": {"wheat", "cereals containing gluten"},
    "dairy": {"milk"},
    "milk": {"milk"},
    "egg": {"eggs"},
    "soy": {"soya"},
    "soya": {"soya"},
    "sesame": {"sesame"},
    "shellfish": {"crustaceans", "molluscs"},
    "crustacean": {"crustaceans"},
    "fish": {"fish"},
    "celery": {"celery"},
    "mustard": {"mustard"},
    "sulphite": {"sulphites"},
    "lupin": {"lupin"},
}
ALLOWED_ALLERGENS = sorted(set().union(*ALLERGEN_VOCAB.values()))

_cat_props = ["category", "category_path", "item_type"]
MENU_CATEGORIES_SET: set[str] = set()
_parent_groups: dict[str, set[str]] = {}
for o in kb.iterator(return_properties=_cat_props):
    props = o.properties
    if props.get("item_type") != "menu_item":
        continue
    category = props.get("category")
    if not isinstance(category, str):
        continue
    MENU_CATEGORIES_SET.add(category)
    path = props.get("category_path")
    if isinstance(path, list) and len(path) >= 2 and isinstance(path[0], str):
        _parent_groups.setdefault(path[0], set()).add(category)
MENU_CATEGORIES = sorted(MENU_CATEGORIES_SET)

# Sibling categories sharing an immediate parent in category_path (e.g. bao buns / gyoza /
# lighter bites / big flavour bites are all "sides > X" -- this menu's own closest thing to a
# starters section, never labelled that anywhere). If the model gets one sibling right, this
# expands category_hint to the rest -- more reliable than expecting it to enumerate every
# sibling from memory, and it can't drift since it's read from the corpus's own structure.
CATEGORY_SIBLINGS: dict[str, set[str]] = {}
for _siblings in _parent_groups.values():
    for _leaf in _siblings:
        CATEGORY_SIBLINGS[_leaf] = _siblings


def expand_category_hint(category_hint: list[str]) -> list[str]:
    """Add sibling categories (same category_path parent) to a guessed category_hint."""
    expanded = set(category_hint)
    for c in category_hint:
        expanded |= CATEGORY_SIBLINGS.get(c, set())
    return sorted(expanded)


UNDERSTAND_SYSTEM_PROMPT = f"""
You turn one guest question for a restaurant chatbot into a structured query-understanding
result used to search a knowledge base. Return only the JSON described by the response
schema -- no extra text.

Fields:
- intent: "menu" if the guest is asking about a dish, ingredient, price, or nutrition value.
  "faq" if they're asking about restaurant policy -- hours, bookings, delivery, payments, gift
  cards, or where to find allergen information.
- dietary: "vegan" or "vegetarian" ONLY when the guest wants dishes filtered to that
  restriction (e.g. "a vegan curry", "vegetarian mains"). Use "none" for a general
  availability question like "do you have vegan options" -- that should still search
  everything rather than be filtered down, since FAQ rows about dietary options carry no
  dietary_tags of their own and a hard filter would hide them.
- price_max_gbp: a number ONLY when the guest gives a firm ceiling ("under £8", "less than
  £10"). null for vague wording like "affordable" or "cheap".
- allergens_exclude: canonical allergen names (from the list below) the guest wants excluded,
  ONLY when they state an allergy, intolerance, or something to avoid (e.g. "I have a nut
  allergy", "dairy-free options", "no shellfish"). Map colloquial terms to every matching
  canonical value -- "nuts" maps to every tree-nut entry plus peanuts, "dairy" maps to milk,
  "shellfish" maps to crustaceans and molluscs. Empty array if no allergy was stated.
- search_query: the question rewritten as a short search phrase for just the dish/food itself
  -- strip out anything already captured by dietary, price_max_gbp, or allergens_exclude
  above (don't repeat "vegan", "under £6", or allergy wording) and strip filler words ("a",
  "do you have", "what's in"). Examples: "a vegan starter under £6" -> "starter";
  "a spicy noodle dish under £8" -> "spicy noodle dish"; "what time do you open" -> "what
  time do you open" (nothing to strip for an FAQ question). Never return an empty string --
  fall back to the original question if there is nothing else to extract.
- category_hint: zero or more names from the menu category list below that best match any
  course-type language in the question (e.g. "starter", "small plate", "main", "dessert",
  "drink") -- the guest's word for a course type rarely matches this menu's own category
  names exactly (there is no category literally called "starters"), so use your judgement
  about which real categories a guest asking for that course type would actually mean.
  Leave empty if the question already names a specific dish, or names no course type, or is
  an "faq" question (this list is menu categories only).
- gluten_free_only: true ONLY when the guest asks for the restaurant's own gluten-free
  menu/section by name ("what's on your gluten-free menu", "gluten-free options"). This is a
  positive filter for that curated section -- separate from allergens_exclude, which is the
  safety exclusion for a guest describing an actual allergy or intolerance. Both can be true
  together (e.g. "I'm coeliac, what's on the gluten-free menu").
- kcal_max: a calorie ceiling as a number. Honour both a firm number ("under 500 calories")
  and qualitative wording ("a low-calorie main" -> use a sensible reference like 500). null
  if calories were not mentioned at all.
- protein_min_g: a protein floor in grams as a number. Honour both a firm number ("at least
  20g protein") and qualitative wording ("a high-protein dish" -> use a sensible reference
  like 20). null if protein was not mentioned at all.
- alcohol_free: true ONLY when the guest explicitly wants a non-alcoholic / alcohol-free
  drink.

Canonical allergens: {", ".join(ALLOWED_ALLERGENS)}
Menu categories: {", ".join(MENU_CATEGORIES)}
""".strip()

RESPONSE_SCHEMA = {
    "type": "OBJECT",
    "properties": {
        "intent": {"type": "STRING", "enum": ["menu", "faq"]},
        "dietary": {"type": "STRING", "enum": ["vegan", "vegetarian", "none"]},
        "price_max_gbp": {"type": "NUMBER", "nullable": True},
        "allergens_exclude": {
            "type": "ARRAY",
            "items": {"type": "STRING", "enum": ALLOWED_ALLERGENS},
        },
        "search_query": {"type": "STRING"},
        "category_hint": {
            "type": "ARRAY",
            "items": {"type": "STRING", "enum": MENU_CATEGORIES},
        },
        "gluten_free_only": {"type": "BOOLEAN"},
        "kcal_max": {"type": "NUMBER", "nullable": True},
        "protein_min_g": {"type": "NUMBER", "nullable": True},
        "alcohol_free": {"type": "BOOLEAN"},
    },
    "required": [
        "intent",
        "dietary",
        "price_max_gbp",
        "allergens_exclude",
        "search_query",
        "category_hint",
        "gluten_free_only",
        "kcal_max",
        "protein_min_g",
        "alcohol_free",
    ],
}


def understand_query(question: str) -> dict:
    """Single LLM call: classify intent and extract retrieval filters from the question."""
    if not LLM_API_KEY:
        return {
            "intent": "menu",
            "dietary": None,
            "price_max_gbp": None,
            "allergens_exclude": [],
            "search_query": question,
            "category_hint": [],
            "gluten_free_only": False,
            "kcal_max": None,
            "protein_min_g": None,
            "alcohol_free": False,
            "usage": _zero_usage(),
        }
    resp = call_llm(UNDERSTAND_SYSTEM_PROMPT, question, response_schema=RESPONSE_SCHEMA)
    parsed = json.loads(resp["text"])
    dietary = parsed.get("dietary") or "none"
    allergens = [a for a in parsed.get("allergens_exclude", []) if a in ALLOWED_ALLERGENS]
    category_hint = [c for c in parsed.get("category_hint", []) if c in MENU_CATEGORIES]
    category_hint = expand_category_hint(category_hint)
    search_query = (parsed.get("search_query") or "").strip() or question
    return {
        "intent": parsed.get("intent") if parsed.get("intent") in ("menu", "faq") else "menu",
        "dietary": None if dietary == "none" else dietary,
        "price_max_gbp": parsed.get("price_max_gbp"),
        "allergens_exclude": allergens,
        "search_query": search_query,
        "category_hint": category_hint,
        "gluten_free_only": bool(parsed.get("gluten_free_only")),
        "kcal_max": parsed.get("kcal_max"),
        "protein_min_g": parsed.get("protein_min_g"),
        "alcohol_free": bool(parsed.get("alcohol_free")),
        "usage": resp["usage"],
    }

### 3.1 Smoke test

`understand_query()` output on a handful of questions -- dietary, price, allergy, a
general-availability question that should NOT get hard-filtered, an FAQ question, and one for
each of the newer fields (`gluten_free_only`, `kcal_max`, `protein_min_g`, `alcohol_free`) --
before it's wired into retrieval.

In [ ]:
for q in [
    "a vegan noodle dish under £10",
    "I have a nut allergy, what can I eat",
    "do you have vegan options",
    "gluten free chicken",
    "what time do you open",
    "what's on your gluten-free menu",
    "a low-calorie main",
    "a high-protein noodle dish",
    "a non-alcoholic drink",
]:
    u = understand_query(q)
    print(f"{q!r}")
    print(f"   {u}")

## 4. Retrieval pipeline

The rest of `01`'s pipeline, unchanged in substance -- only `parse_constraints()` is gone,
replaced by two functions consuming `understand_query()`'s output: `build_filter()` (dietary +
price -> a Weaviate server-side filter, as before) and `build_search_text()` (new -- see
below). `FIELDS` still carries the two additions generation needs: `description` (the actual
grounding text, not just the vectorized `embedding_text`) and `kcal`.

`search()` now retrieves and reranks against `build_search_text(u)`, not the raw question --
`understand_query()`'s cleaned `search_query` plus its `category_hint` appended as plain
text. `search()`'s result dict carries `search_text` too, so it's visible in every trace
alongside `understanding`, not just used silently.

`rerank()` now also returns Cohere's own `meta.billed_units.search_units` for its call --
real billing data (one search unit per up to 100 documents), not an estimate, zero on the
hybrid-order fallback since nothing billable happened. This is the only embedding-side cost
this notebook can actually observe: the hybrid query's own vectorization of your question text
happens *inside* Weaviate's managed `text2vec_cohere` integration, and Weaviate's query
response never reports what that internal Cohere call cost -- there is no client-visible
number for it with this architecture. `show_answer()` (section 8) reports the rerank units
per question and running-session total on that basis, not a full embedding cost.

`rerank()` now paces itself under `COHERE_MAX_RPM` too, the same proactive-pacing idea section
2 already applies to Gemini -- previously only the LLM side had this, but constraint
relaxation below means one question can now trigger several `rerank()` calls instead of
exactly one, which raises the odds of a Cohere 429 the same way multiple LLM calls did before
`_pace_llm_call()` existed.

**Constraint relaxation** -- borrowed from a similar RAG assignment
(`C1M5_Assignment_Solve.ipynb`, which relaxes clothing-store filters like colour and category
when a search comes up too thin). If nothing clears `GATE` on the first attempt, `search()`
drops one `RELAXABLE_FIELDS` constraint at a time (`kcal_max`, `protein_min_g`,
`gluten_free_only`, `alcohol_free`, then `price_max_gbp` last) and retries, instead of
declining outright when a slightly-off-spec match exists. Adapted for a restaurant rather than
copied outright: `dietary` and `allergens_exclude` are **never** in `RELAXABLE_FIELDS` and
never get cleared -- the clothing example relaxes every filter including gender and category,
which is fine for "no exact colour match" but would be actively harmful here (silently
suggesting a non-vegan dish to a vegan guest, or one containing a stated allergen). The
returned `relaxed_fields` list feeds into section 7's `build_user_prompt()`, so the model is
told explicitly which constraint was dropped and states the dish's real figure instead of
implying a false match. `rerank_search_units` accumulates across every relaxation attempt,
not just the last one, since each retry is a genuine extra Cohere call.

In [ ]:
ALPHA = 0.75
K = 20
TOP_N = 6
GATE = 0.15
RERANK_MODEL = "rerank-v3.5"
COHERE_MAX_RPM = 15.0  # client-side pacing cap for rerank() -- edit to match your key's quota

_last_cohere_call_at = 0.0


def _pace_cohere_call() -> None:
    """Block just long enough to keep rerank() under COHERE_MAX_RPM requests/minute."""
    global _last_cohere_call_at
    min_interval = 60.0 / COHERE_MAX_RPM
    wait = min_interval - (time.monotonic() - _last_cohere_call_at)
    if wait > 0:
        time.sleep(wait)
    _last_cohere_call_at = time.monotonic()


FIELDS = [
    "name",
    "category",
    "item_type",
    "description",
    "price_gbp",
    "kcal",
    "protein_g",
    "abv_percent",
    "dietary_tags",
    "allergens_contains",
    "allergens_may_contain",
    "is_gluten_free_listed",
    "embedding_text",
]


def pstr(o, key: str) -> str:
    """properties[key] as a string, or "" if it is not one."""
    v = o.properties.get(key)
    return v if isinstance(v, str) else ""


def plist(o, key: str) -> list:
    """properties[key] as a list, or [] if it is not one."""
    v = o.properties.get(key)
    return v if isinstance(v, list) else []


def pnum(o, key: str) -> float | None:
    """properties[key] as a float, or None if it is not numeric."""
    v = o.properties.get(key)
    return float(v) if isinstance(v, (int, float)) else None


def allergen_set(o) -> set[str]:
    """Union of an object's allergens_contains and allergens_may_contain."""
    return {*plist(o, "allergens_contains"), *plist(o, "allergens_may_contain")}


def build_filter(u: dict) -> FilterReturn | None:
    """Turn an understand_query() result into a Weaviate server-side filter."""
    clauses: list[FilterReturn] = []
    if u["dietary"]:
        clauses.append(Filter.by_property("dietary_tags").contains_any([u["dietary"]]))
    if u["price_max_gbp"] is not None:
        clauses.append(Filter.by_property("price_gbp").less_or_equal(float(u["price_max_gbp"])))
    if u["gluten_free_only"]:
        clauses.append(Filter.by_property("is_gluten_free_listed").equal(True))
    if u["kcal_max"] is not None:
        clauses.append(Filter.by_property("kcal").less_or_equal(float(u["kcal_max"])))
    if u["protein_min_g"] is not None:
        clauses.append(Filter.by_property("protein_g").greater_or_equal(float(u["protein_min_g"])))
    if u["alcohol_free"]:
        clauses.append(Filter.by_property("abv_percent").is_none(True))
    return Filter.all_of(clauses) if clauses else None


def build_search_text(u: dict) -> str:
    """Turn an understand_query() result into the text used for hybrid retrieval + rerank.

    Uses search_query (dietary/price/allergy wording already stripped) instead of the raw
    question, so those tokens stop diluting the vector match once build_filter() has already
    handled them deterministically. category_hint is appended as a soft signal -- it's plain
    text, not a hard filter, so a wrong guess can only fail to help, never hide the right row.
    """
    text = u["search_query"]
    if u["category_hint"]:
        text = f"{text} ({', '.join(u['category_hint'])})"
    return text


def retrieve(query: str, k: int = K, alpha: float = ALPHA, filters: FilterReturn | None = None):
    """Run one hybrid (keyword + vector) query against KnowledgeBase."""
    return kb.query.hybrid(
        query=query,
        alpha=alpha,
        limit=k,
        filters=filters,
        return_properties=FIELDS,
        return_metadata=MetadataQuery(score=True),
    ).objects


def rerank(query: str, objs: list, top_n: int = TOP_N) -> tuple[list[dict], int]:
    """Rerank objs against query with Cohere; fall back to hybrid order on rate limit.

    Paces itself under COHERE_MAX_RPM before every attempt -- constraint relaxation in
    search() can call this more than once per question, so this needs the same proactive
    pacing call_llm() has, not just the existing reactive 429 retry below. Returns
    (ranked_hits, search_units) -- search_units is Cohere's own meta.billed_units.search_units
    for this call (real billing data, not an estimate); 0 on the hybrid-order fallback, since
    no billable rerank call actually completed.
    """
    if not objs:
        return [], 0
    docs = [pstr(o, "embedding_text") or pstr(o, "name") for o in objs]
    payload = json.dumps(
        {"model": RERANK_MODEL, "query": query, "documents": docs, "top_n": min(top_n, len(docs))}
    ).encode()
    results = None
    search_units = 0
    for attempt in range(4):
        _pace_cohere_call()
        try:
            req = urllib.request.Request(
                "https://api.cohere.com/v2/rerank",
                data=payload,
                headers={
                    "Authorization": f"Bearer {COHERE_KEY.strip()}",
                    "Content-Type": "application/json",
                },
                method="POST",
            )
            with urllib.request.urlopen(req, timeout=30) as resp:
                data = json.load(resp)
            results = data["results"]
            search_units = data.get("meta", {}).get("billed_units", {}).get("search_units", 0)
            break
        except urllib.error.HTTPError as e:
            retryable = e.code == 429 and attempt < 3
            code_ = e.code
            e.close()
            if retryable:
                time.sleep(2 * 4**attempt)
            else:
                print(f"   [rerank HTTP {code_}; using hybrid order]")
                break
    if results is None:
        hits = [
            {"obj": o, "rerank": math.nan, "hybrid": o.metadata.score or 0.0} for o in objs[:top_n]
        ]
        return hits, 0
    hits = [
        {
            "obj": objs[r["index"]],
            "rerank": float(r["relevance_score"]),
            "hybrid": objs[r["index"]].metadata.score or 0.0,
        }
        for r in results
    ]
    return hits, search_units


RELAXABLE_FIELDS = [
    "kcal_max",
    "protein_min_g",
    "gluten_free_only",
    "alcohol_free",
    "price_max_gbp",
]


def _is_constraint_set(u: dict, field: str) -> bool:
    return bool(u[field]) if field in ("gluten_free_only", "alcohol_free") else u[field] is not None


def _clear_constraint(u: dict, field: str) -> dict:
    """A copy of u with one relaxable constraint cleared back to its unset value."""
    cleared = dict(u)
    cleared[field] = False if field in ("gluten_free_only", "alcohol_free") else None
    return cleared


def search(question: str, gate: float = GATE) -> dict:
    """Full pipeline: understand the query (LLM), retrieve, exclude allergens, rerank, gate.

    If nothing clears the gate, progressively drops one RELAXABLE_FIELDS constraint at a time
    (least essential first) and retries, rather than declining outright when a closer, slightly
    off-spec match exists. dietary and allergens_exclude are never relaxed -- those are safety
    and preference guarantees, not refinements, and silently dropping either could put an unsafe
    or unwanted dish in front of a guest.
    """
    u = understand_query(question)
    current = u
    relaxed_fields: list[str] = []
    total_search_units = 0
    while True:
        server = build_filter(current)
        exclude = set(current["allergens_exclude"])
        search_text = build_search_text(current)
        objs = retrieve(search_text, filters=server)
        kept = [o for o in objs if not (allergen_set(o) & exclude)] if exclude else objs
        ranked, search_units = rerank(search_text, kept)
        total_search_units += search_units
        top = ranked[0]["rerank"] if ranked else 0.0
        answerable = bool(ranked) if math.isnan(top) else top >= gate
        if answerable:
            break
        next_field = next((f for f in RELAXABLE_FIELDS if _is_constraint_set(current, f)), None)
        if next_field is None:
            break
        relaxed_fields.append(next_field)
        current = _clear_constraint(current, next_field)
    return {
        "understanding": u,
        "relaxed_fields": relaxed_fields,
        "search_text": search_text,
        "excluded": sorted(exclude),
        "retrieved": len(objs),
        "kept": len(kept),
        "ranked": ranked,
        "top": top,
        "answerable": answerable,
        "rerank_search_units": total_search_units,
    }


def line(o) -> str:
    """One-line summary of an object: type, name, category, price, dietary tags."""
    price = pnum(o, "price_gbp")
    price_s = f"  £{price:.2f}" if price is not None else ""
    diet = plist(o, "dietary_tags")
    diet_s = f"  {diet}" if diet else ""
    return f"[{pstr(o, 'item_type')}] {pstr(o, 'name')} <{pstr(o, 'category')}>{price_s}{diet_s}"

## 5. Tone policy

This section used to be "Temperature policy": `TEMP_MENU = 0.2` / `TEMP_FAQ = 0.8` passed to
`call_llm()`. It did nothing -- Gemini's 3.x Flash line (`gemini-3.8-flash` included) silently
ignores `temperature`/`top_p`/`top_k`, confirmed against Google's own developer docs and
forum: the API accepts the parameter, returns `200`, and never applies it, with future model
generations rejecting it outright (HTTP 400) instead. Google's own migration guidance is to
steer behavior through system-instruction wording instead of sampling parameters, which is
what `tone_for()` does now -- same intent-based split as before, delivered as prompt text
instead of a request parameter that the model was never reading.

- **Menu** -- precise and literal. Price, allergens, and nutrition are facts pulled from
  `CONTEXT`; the instruction tells the model to stay close to `CONTEXT`'s exact wording rather
  than paraphrase a number or an allergen into something wrong.
- **FAQ** -- warm and conversational. House-policy answers (hours, bookings, payments) are
  copy, not numbers; the instruction gives it room to phrase things naturally as long as the
  substance still matches `CONTEXT`.

In [ ]:
MENU_TONE = (
    "Tone for this answer: precise and literal. This is a factual menu question -- stick "
    "closely to CONTEXT's exact wording for prices, allergens, dietary tags, and nutrition "
    "figures. Do not paraphrase or round a number, and do not add warmth or small talk that "
    "risks softening a factual claim."
)
FAQ_TONE = (
    "Tone for this answer: warm and conversational. This is a house-policy question -- feel "
    "free to phrase the answer naturally, in your own words, as long as the substance matches "
    "CONTEXT exactly."
)


def tone_for(intent: str) -> str:
    return FAQ_TONE if intent == "faq" else MENU_TONE

## 6. System instructions (generation)

The grounding rules sent as the model's system instruction on every generation call -- fixed
text, independent of intent (section 8 appends `tone_for(intent)` to it per call), and
distinct from `UNDERSTAND_SYSTEM_PROMPT` above (that one extracts filters; this one answers
the guest). It encodes the project's own
data caveats directly: `price_gbp` is mixed real/estimated with no per-row marker (see this
project's documented **Data provenance**), allergen safety needs the `contains`/`may_contain` union,
and the `(gluten-free recipe)` / `(vegan recipe)` suffix marks a genuinely different dish, not
a duplicate.

In [ ]:
GENERATION_SYSTEM_PROMPT = """
You are the menu assistant for a restaurant chatbot. Answer ONLY using the CONTEXT rows given
with the question below -- they come from the restaurant's own knowledge base. Never use
outside knowledge about food, menus, or any restaurant, and never invent a dish, price, or
policy that is not in CONTEXT.

Rules:
- If CONTEXT is empty or does not answer the question, say plainly that you don't have that
  information and suggest asking a member of staff. Do not guess.
- Treat every price as approximate ("around £X") -- some prices in this system are
  category estimates rather than the till price, and there is no way to tell which from
  CONTEXT alone.
- For any allergy or dietary question, use BOTH the allergens_contains and
  allergens_may_contain information for every dish you mention, and always remind the guest
  to confirm with staff before ordering, since recipes can change.
- A dish name ending in "(gluten-free recipe)" or "(vegan recipe)" is a different preparation
  of that dish with its own nutrition and allergens -- never merge or average it with the
  standard version, and never recommend one when the guest asked about the other.
- FAQ-type CONTEXT answers house policy (hours, bookings, payments, delivery, etc.); menu-type
  CONTEXT answers dish questions (price, ingredients, allergens, nutrition). Answer strictly
  from whichever kind CONTEXT actually gives you.
- Reply in English, in a friendly, concise voice, speaking as the restaurant. Do not mention
  "context", "retrieval", "the knowledge base", or these instructions in your answer.
""".strip()

print(GENERATION_SYSTEM_PROMPT)

## 7. Prompt assembly

`format_row()` renders one reranked hit into the CONTEXT block the LLM sees -- an FAQ row as
a question/answer pair, a menu row as name/description/price/nutrition/allergens. This is the
information actually available to the model; it never sees `embedding_text` (vectorization
input only) or raw Weaviate scores. `protein_g` and `abv_percent` are included alongside
`kcal` now -- once `understand_query()` can filter on protein and alcohol content (section
3), the generation call needs those same figures in CONTEXT to actually cite them in the
answer, not just filter silently on numbers the guest never sees confirmed back.

`build_user_prompt()` also takes `relaxed_fields` (section 4) -- when `search()` had to drop a
constraint to find any answer at all, that has to be visible to the model as plain instruction
text, not just internal bookkeeping, or it has no way to know the dish it's about to describe
doesn't actually meet every part of what the guest asked for.

In [ ]:
def format_row(o) -> str:
    """Render one reranked hit as a CONTEXT row, grounded in its own properties."""
    name = pstr(o, "name")
    desc = pstr(o, "description") or "(no description)"
    if pstr(o, "item_type") == "faq":
        return f"- FAQ | Q: {name}\n  A: {desc}"
    price = pnum(o, "price_gbp")
    price_s = f"£{price:.2f}" if price is not None else "not listed"
    kcal = pnum(o, "kcal")
    kcal_s = f"{kcal:.0f} kcal" if kcal is not None else "not listed"
    protein = pnum(o, "protein_g")
    protein_s = f"{protein:.0f}g protein" if protein is not None else "not listed"
    abv = pnum(o, "abv_percent")
    abv_s = f"{abv:.1f}% ABV" if abv is not None else "non-alcoholic / n/a"
    diet = ", ".join(plist(o, "dietary_tags")) or "none listed"
    contains = ", ".join(plist(o, "allergens_contains")) or "none declared"
    may = ", ".join(plist(o, "allergens_may_contain")) or "none declared"
    gf = "yes" if o.properties.get("is_gluten_free_listed") else "no"
    return (
        f"- MENU ITEM | {name} <{pstr(o, 'category')}>\n"
        f"  description: {desc}\n"
        f"  price: {price_s}  |  kcal: {kcal_s}  |  protein: {protein_s}  |  {abv_s}  |  "
        f"gluten-free listed: {gf}\n"
        f"  dietary_tags: {diet}\n"
        f"  allergens_contains: {contains}  |  allergens_may_contain: {may}"
    )


def build_context(ranked: list[dict]) -> str:
    """CONTEXT block handed to the LLM: one formatted row per reranked hit."""
    if not ranked:
        return "(no matching rows retrieved)"
    return "\n".join(format_row(h["obj"]) for h in ranked)


def build_user_prompt(question: str, context: str, relaxed_fields: list[str]) -> str:
    """The full user-turn text sent to the LLM alongside GENERATION_SYSTEM_PROMPT.

    When search() had to relax a constraint to find any answerable match, that has to reach
    the model explicitly -- otherwise it has no way to know a shown dish doesn't actually meet
    every part of the original ask, and could misreport it as a full match.
    """
    text = f"QUESTION: {question}\n\nCONTEXT:\n{context}"
    if relaxed_fields:
        text += (
            f"\n\nNOTE: no result matched every part of the question. To surface a closest "
            f"match, these constraints were dropped: {', '.join(relaxed_fields)}. Be upfront "
            f"that the dish doesn't fully satisfy {', '.join(relaxed_fields)} -- state its "
            f"actual figure from CONTEXT rather than implying it meets the original ask."
        )
    return text

## 8. End-to-end answer function

`answer()` is the full path: `search()` (LLM understanding + retrieval) -> `tone_for()` from
the understanding step's intent, appended to `GENERATION_SYSTEM_PROMPT` -> prompt assembly ->
`call_llm()` for generation -- and
returns every intermediate artifact, not just the final text. `show_answer()` prints all of
it: what the understanding call extracted, the retrieval verdict, the exact system and user
prompts sent to the generation call, the answer, and what this question actually cost on
**both** APIs the pipeline touches:

- **LLM (Gemini)** -- understanding and generation token counts, separately, from each call's
  own `usageMetadata` (not estimated).
- **Embedding (Cohere)** -- the rerank call's `search_units`, real billing data from section
  4's `rerank()`. The hybrid query's own vectorization of your question text is *not*
  included -- it happens inside Weaviate's managed integration and Weaviate never reports
  what that internal Cohere call cost, so there is nothing to read for it from this notebook.

Both roll up into `SESSION_USAGE`, so a whole "Run All" shows its cumulative consumption
across every test cell, not just the last question's.

In [ ]:
SESSION_USAGE = {"questions": 0, "total_tokens": 0, "rerank_search_units": 0}


def answer(question: str) -> dict:
    """Full pipeline: search (understand + retrieve) -> build prompt -> generate."""
    result = search(question)
    intent = result["understanding"]["intent"]
    tone = tone_for(intent)
    system_prompt = f"{GENERATION_SYSTEM_PROMPT}\n\n{tone}"
    context = build_context(result["ranked"]) if result["answerable"] else "(no confident match)"
    user_prompt = build_user_prompt(question, context, result["relaxed_fields"])
    if not LLM_API_KEY:
        reply = "[LLM_API_KEY not set -- skipping live call]"
        gen_usage = _zero_usage()
    else:
        gen = call_llm(system_prompt, user_prompt)
        reply = gen["text"]
        gen_usage = gen["usage"]
    understand_usage = result["understanding"]["usage"]
    usage = {
        "understand": understand_usage,
        "generate": gen_usage,
        "total_tokens": understand_usage["total_tokens"] + gen_usage["total_tokens"],
    }
    return {
        "question": question,
        "search": result,
        "intent": intent,
        "tone": tone,
        "system_prompt": system_prompt,
        "user_prompt": user_prompt,
        "answer": reply,
        "usage": usage,
    }


def show_answer(question: str) -> dict:
    """Run answer() and print the full trace: understanding, retrieval, prompts, and reply."""
    r = answer(question)
    s = r["search"]
    u = s["understanding"]
    usage = r["usage"]
    rerank_units = s["rerank_search_units"]
    SESSION_USAGE["questions"] += 1
    SESSION_USAGE["total_tokens"] += usage["total_tokens"]
    SESSION_USAGE["rerank_search_units"] += rerank_units
    verdict = "ANSWERABLE" if s["answerable"] else "NO CONFIDENT MATCH"
    print(f"q: {question!r}")
    print(
        f"   understanding: intent={u['intent']}  dietary={u['dietary']}  "
        f"price_max_gbp={u['price_max_gbp']}  allergens_exclude={u['allergens_exclude'] or '-'}  "
        f"category_hint={u['category_hint'] or '-'}"
    )
    print(
        f"   gluten_free_only={u['gluten_free_only']}  kcal_max={u['kcal_max']}  "
        f"protein_min_g={u['protein_min_g']}  alcohol_free={u['alcohol_free']}"
    )
    print(f"   search_text: {s['search_text']!r}")
    print(
        f"   retrieval={verdict} (top rr {s['top']:.3f})  "
        f"retrieved {s['retrieved']} -> kept {s['kept']}"
        + (f"  relaxed={s['relaxed_fields']}" if s["relaxed_fields"] else "")
    )
    print(
        f"   LLM usage: understand {usage['understand']['total_tokens']} tok + "
        f"generate {usage['generate']['total_tokens']} tok = {usage['total_tokens']} tok "
        f"for this question  (session so far: {SESSION_USAGE['total_tokens']} tok over "
        f"{SESSION_USAGE['questions']} questions)"
    )
    print(
        f"   embedding usage: rerank {rerank_units} search unit(s) this question  "
        f"(session so far: {SESSION_USAGE['rerank_search_units']}); query vectorization runs "
        f"inside Weaviate and isn't reported back to the client -- not counted here"
    )
    print("\n--- SYSTEM PROMPT (generation) ---")
    print(r["system_prompt"])
    print("\n--- USER PROMPT ---")
    print(r["user_prompt"])
    print("\n--- ANSWER ---")
    print(r["answer"])
    print()
    return r

## 9. Menu question tests (precise tone)

Dish questions -- price, ingredients, dietary tags, and the newer nutrition/section filters --
should get `intent=menu` from the understanding call and run with `MENU_TONE`.

In [ ]:
for q in [
    "what's in the vegan ramen and how much is it",
    "a spicy noodle dish under £8",
    "how many calories are in the katsu curry",
    "what desserts do you have",
    "a high-protein main that's also gluten-free listed",
]:
    show_answer(q)

## 10. FAQ question tests (warm tone)

House-policy questions should get `intent=faq` from the understanding call and run with
`FAQ_TONE`.

In [ ]:
for q in [
    "what time do you open",
    "can I book a table for a large group",
    "where can I find allergen information",
    "do you sell gift cards",
]:
    show_answer(q)

## 11. Edge cases

The three hazards `01` identified, carried through understanding and generation: a stated
allergy must exclude both allergen lists before the LLM ever sees the row, the two same-name
recipe variants must stay distinguishable rather than merged, and an out-of-domain question
must be declined, not answered from the model's own knowledge.

In [ ]:
show_answer("I have a nut allergy, what noodle dishes are safe for me")

_vp = ["name"]
variant_names = [
    pstr(o, "name") for o in kb.iterator(return_properties=_vp) if "recipe)" in pstr(o, "name")
]
base = variant_names[0].split(" (")[0] if variant_names else "signature seafood ramen"
show_answer(f"what's the difference between the gluten-free and vegan {base}")

show_answer("how do I change a car tyre")

## 12. Recommendations carried into `app/`

Combining `01`'s retrieval recommendations with what this notebook adds:

- Query understanding: one structured-output LLM call for intent + filters, run before
  retrieval -- not a post-hoc guess from whichever row ranked first, and not the regex
  heuristics `01` prototyped with. Constrain free-form extraction (like `allergens_exclude`)
  with a `response_schema` enum tied to the corpus's own vocabulary, so the model can't return
  a value that will never match anything.
- Retrieval: hybrid `alpha=0.75` -> allergen exclude (union of `allergens_contains`/
  `allergens_may_contain`) -> rerank `rerank-v3.5` -> gate `0.15`.
- Tone by intent via system-prompt text, not a `temperature` value -- Gemini's 3.x Flash line
  silently ignores `temperature`/`top_p`/`top_k`, confirmed against Google's own docs and
  forum, so a numeric per-intent knob would have been a no-op. If `app/` ever moves to a
  model/provider where sampling parameters do work, `tone_for()`'s split (precise for menu
  facts, warm for FAQ copy) is still the right idea -- just re-verify whether it belongs as a
  parameter or a prompt instruction for whatever model is actually in use, rather than
  assuming either.
- One fixed, grounding-only system prompt for generation; the per-question CONTEXT is the
  only thing that changes between calls. Keeps the price-estimate, allergen-union, and
  documented variant-recipe caveats enforced at the prompt level, not just in code.
- A failed gate must reach the LLM as "(no confident match)" in CONTEXT, never as an empty
  prompt -- the system prompt's decline-don't-guess rule only fires if CONTEXT actually says
  there is nothing to work with.
- Two LLM calls per question (understand, then generate) cost more latency than the regex
  version -- worth it here because understanding replaces several brittle heuristics at once;
  worth re-measuring against real traffic before assuming it's always the right trade.
- Every outbound LLM call needs retry with backoff -- a live 503 mid-test-run (section 3.1)
  is what motivated `call_llm()`'s full-jitter retry logic. `app/`'s retrieval tool will make
  the same two calls per guest turn, so the same retry wrapper (or a LangGraph-native
  equivalent) has to travel with it, not just live in this notebook.
- Don't retrieve against the raw question once an LLM has already parsed it -- a live run of
  "a vegan starter under £6" returned NO CONFIDENT MATCH despite a real match existing
  (`data/knowledge_base.json` has exactly one: a bao bun at £5.90), because "vegan" and
  "under £6" diluted the vector match and this corpus has no category literally called
  "starters". `search_query` (dietary/price/allergy wording stripped) and `category_hint`
  (guest course-type language mapped to the menu's own category names, fetched live so it
  can't drift) fixed it in this notebook -- carry both into `app/`, and keep `category_hint`
  a soft signal folded into search text rather than a hard filter, since a wrong guess should
  only fail to help, never hide the right dish.

## 13. Demo -- ask your own question

Edit `QUESTION` and re-run this cell as many times as you like -- it reuses the connection
opened in cell 1, so there's no need to re-run the notebook from the top.

In [ ]:
QUESTION = "a vegan starter under £6"  # <- edit this, then re-run the cell

_ = show_answer(QUESTION)